# Week 3: V1 Data Pipeline (TalentRank)

**Project:** TalentRank: Explainable Machine Learning for Employee Promotion Prediction and Ranking
**Pathway:** B — CSV (structured tabular data)
**Run order (Topic 3 lab):** Cell 1 → Cell 3 → A3 inspection → Cell 4 → Cell 4A → Cell 5B → B2 comparison → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8B → Cell 9

Run from the `notebooks/` folder of the repository, or in Google Colab after uploading `employee_promotion.csv` to `/content/week3_project/data/raw/`.

## Part A: Start with the data
### Cell 1: Workspace setup

In [45]:
# Workspace setup (RUN FIRST)
import sys
from pathlib import Path

# Google Colab uses the lab folder; locally the repository root is used
if "google.colab" in sys.modules:
    BASE_DIR = Path("/content/week3_project")
else:
    BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR.relative_to(BASE_DIR).as_posix())
print("Processed folder:", PROCESSED_DIR.relative_to(BASE_DIR).as_posix())

Raw folder: data/raw
Processed folder: data/processed


### Verify inputs

In [46]:
# Verify uploaded inputs
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR).as_posix(), "bytes:", f.stat().st_size)
assert files, "Add employee_promotion.csv to data/raw before proceeding."
print("Input verified.")

Input count: 1
employee_promotion.csv bytes: 3644896
Input verified.


### Step A3: Inspect the selected data source

Requirement link: **FR1** (inspect records) and **FR3** (labelled data for prediction). The inspection unit is one employee record. Two representative subsets are inspected: promoted vs not-promoted employees, and records with missing values.

In [47]:
import pandas as pd

raw_df = pd.read_csv(RAW_DIR / "employee_promotion.csv")
print("Rows, columns:", raw_df.shape)
print("\nColumns and types:")
print(raw_df.dtypes.astype(str).to_string())

print("\nSubset 1a — five promoted employees:")
print(raw_df[raw_df["is_promoted"] == 1].head().to_string(index=False))
print("\nSubset 1b — five not-promoted employees:")
print(raw_df[raw_df["is_promoted"] == 0].head().to_string(index=False))
print("\nSubset 2 — five records with at least one missing value:")
print(raw_df[raw_df.isna().any(axis=1)].head().to_string(index=False))

Rows, columns: (54808, 13)

Columns and types:
employee_id               int64
department               object
region                   object
education                object
gender                   object
recruitment_channel      object
no_of_trainings           int64
age                       int64
previous_year_rating    float64
length_of_service         int64
awards_won                int64
avg_training_score      float64
is_promoted               int64

Subset 1a — five promoted employees:
 employee_id        department    region        education gender recruitment_channel  no_of_trainings  age  previous_year_rating  length_of_service  awards_won  avg_training_score  is_promoted
       49017 Sales & Marketing  region_7       Bachelor's      f            sourcing                1   35                   5.0                  3           0                50.0            1
       58304 Sales & Marketing region_28       Bachelor's      m            sourcing                1   33       

In [48]:
# Target distribution and missing values
target = raw_df["is_promoted"].value_counts().rename({0: "Not promoted (0)", 1: "Promoted (1)"})
print(pd.DataFrame({"Records": target, "Percent": (target / len(raw_df) * 100).round(2)}))

missing = raw_df.isna().sum()
print("\nMissing values:")
print(pd.DataFrame({"Missing": missing, "Percent": (missing / len(raw_df) * 100).round(2)})[missing > 0])
print("\nExact duplicate rows:", int(raw_df.duplicated().sum()))
print("Duplicate employee_id:", int(raw_df["employee_id"].duplicated().sum()))

# Why is previous_year_rating missing?
miss_rating = raw_df[raw_df["previous_year_rating"].isna()]
print("\nMissing previous_year_rating by length_of_service:")
print(miss_rating["length_of_service"].value_counts().to_string())

                  Records  Percent
is_promoted                       
Not promoted (0)    50140    91.48
Promoted (1)         4668     8.52

Missing values:
                      Missing  Percent
education                2409     4.40
previous_year_rating     4124     7.52
avg_training_score       2560     4.67

Exact duplicate rows: 0
Duplicate employee_id: 0

Missing previous_year_rating by length_of_service:
length_of_service
1    4124


## Part B: Prepare the data
### Source selection and lineage utilities

In [49]:
# REQUIRED: Source Selection + Lineage Utilities
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [p.relative_to(RAW_DIR).as_posix() for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target.relative_to(BASE_DIR).as_posix())
    return manifest

print("Cell 4 completed: select_source(), source_hash(), save_manifest() ready.")

Cell 4 completed: select_source(), source_hash(), save_manifest() ready.


### Common lineage registry

In [50]:
# COMMON LINEAGE REGISTRY
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = {
        "source": source, "output": output, "processing": processing.strip()}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready. Entries:", len(lineage_entries))

Common lineage registry ready. Entries: 0


### Pathway B: prepare the tabular data (Configuration A, selected)

Cleaning rules: remove exact duplicate rows and trim text. Missing values are **kept**: imputation is done later inside model training, fitted on the training split only, to avoid data leakage.

In [51]:
# PATHWAY B — CSV
# Step 1: Select the exact source file
f = select_source(relative_path="employee_promotion.csv")

# Step 2: Load
df = pd.read_csv(f)

# Step 3: Inspect original data
print("--- Original Dataset ---")
print("Source:", f.name)
print("Rows/columns:", df.shape)
print("Missing per column:", {k: int(v) for k, v in df.isna().sum().items() if v})
print("Exact duplicate rows:", int(df.duplicated().sum()))
assert not df.empty, "The selected dataset is empty."

# Step 4: Apply justified cleaning rules (Configuration A)
REMOVE_DUPLICATES = True
TRIM_TEXT = True
prepared = df.copy()
if REMOVE_DUPLICATES:
    prepared = prepared.drop_duplicates().copy()
if TRIM_TEXT:
    for col in prepared.select_dtypes(include=["object", "string"]).columns:
        prepared[col] = prepared[col].astype("string").str.strip()
# Missing values are NOT imputed; target labels and outliers are NOT changed.

# Step 5: Compare before and after preparation
print("\n--- Preparation Comparison ---")
print("Original rows:", len(df), "| Prepared rows:", len(prepared), "| Rows removed:", len(df) - len(prepared))
print("Missing values after:", {k: int(v) for k, v in prepared.isna().sum().items() if v})

# Step 6: Export prepared data (Unix line endings give the same SHA-256 on every OS)
prepared_path = PROCESSED_DIR / "prepared_table.csv"
prepared.to_csv(prepared_path, index=False, lineterminator="\n")

# Step 7: Validate exported data
exported = pd.read_csv(prepared_path)
assert len(exported) == len(prepared), "Exported row count does not match"
assert exported.columns.tolist() == prepared.columns.tolist(), "Exported columns do not match"

print("\nPATHWAY B completed. Prepared shape:", prepared.shape)
print("Saved:", prepared_path.relative_to(BASE_DIR).as_posix())
print("Validation: PASS")

--- Original Dataset ---
Source: employee_promotion.csv
Rows/columns: (54808, 13)
Missing per column: {'education': 2409, 'previous_year_rating': 4124, 'avg_training_score': 2560}
Exact duplicate rows: 0

--- Preparation Comparison ---
Original rows: 54808 | Prepared rows: 54808 | Rows removed: 0
Missing values after: {'education': 2409, 'previous_year_rating': 4124, 'avg_training_score': 2560}

PATHWAY B completed. Prepared shape: (54808, 13)
Saved: data/processed/prepared_table.csv
Validation: PASS


### Compare two preparation configurations

- **Configuration A (selected):** remove duplicates, trim text, keep missing values.
- **Configuration B (alternative):** Configuration A + drop every row that has a missing value.

In [52]:
config_b = prepared.dropna().copy()

def profile(data):
    return {
        "Records": len(data),
        "Missing values": int(data.isna().sum().sum()),
        "Promotion rate (%)": round(data["is_promoted"].mean() * 100, 2),
        "Employees with 1 year of service": int((data["length_of_service"] == 1).sum()),
        "Legal department records": int((data["department"] == "Legal").sum()),
        "Operations department records": int((data["department"] == "Operations").sum()),
    }

comparison = pd.DataFrame({
    "Configuration A - keep missing (selected)": profile(prepared),
    "Configuration B - drop missing rows": profile(config_b),
})
removed = len(prepared) - len(config_b)
one_year_lost = 1 - (config_b["length_of_service"] == 1).sum() / (prepared["length_of_service"] == 1).sum()
print(f"Configuration B removes {removed} rows ({removed / len(prepared):.2%}) "
      f"and {one_year_lost:.2%} of employees with 1 year of service.")
comparison

Configuration B removes 8428 rows (15.38%) and 92.04% of employees with 1 year of service.


,Configuration A - keep missing (selected),Configuration B - drop missing rows
Records,54808.00,46380.00
Missing values,9093.00,0.00
Promotion rate (%),8.52,8.78
Employees with 1 year of service,4547.00,362.00
Legal department records,1039.00,765.00
Operations department records,11348.00,9259.00


**Decision:** Configuration A is selected. Configuration B would remove about 15% of records, almost all employees with one year of service (their `previous_year_rating` is missing because they have no previous year to rate) and about a quarter of Legal records. That would create pipeline-induced bias against new employees and small departments (FR3, RAI1).

### Basic output contract

In [53]:
# BASIC OUTPUT CONTRACT — REQUIRED
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path.relative_to(BASE_DIR).as_posix(), "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty; representative records inspected below.")
print(exported.sample(5, random_state=42).to_string(index=False))

Output: data/processed/prepared_table.csv Size: 3795951
Validation: file exists and is non-empty; representative records inspected below.
 employee_id        department    region  education gender recruitment_channel  no_of_trainings  age  previous_year_rating  length_of_service  awards_won  avg_training_score  is_promoted
        5599                HR region_16 Bachelor's      m               other                1   35                   5.0                 10           0                47.0            0
       73176 Sales & Marketing  region_7 Bachelor's      m               other                3   33                   1.0                  4           0                51.0            0
       13157           Finance  region_2 Bachelor's      m               other                2   38                   3.0                 11           0                64.0            0
        7458        Operations region_22 Bachelor's      f               other                1   33              

### Register Pathway B lineage

In [54]:
# Register Pathway B — CSV
register_lineage(
    source=f,
    output=prepared_path,
    processing=(f"remove_duplicates={REMOVE_DUPLICATES}; trim_text={TRIM_TEXT}; "
                "missing values preserved (no imputation); "
                "rejected alternative: drop rows with missing values"),
)

Registered: employee_promotion.csv -> prepared_table.csv


### Generate the common lineage manifest

In [55]:
# GENERATE COMMON LINEAGE MANIFEST
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file() and output.is_file() and output.stat().st_size > 0
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": source.relative_to(RAW_DIR.resolve()).as_posix(),
        "source_url": "https://www.kaggle.com/datasets/muhammadimran112233/employees-evaluation-for-promotion",
        "source_sha256": source_hash(source),
        "prepared_output": output.relative_to(BASE_DIR.resolve()).as_posix(),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "record_id_field": "employee_id",
        "source_version": "Not verified - collection period and version undocumented",
        "access_rights": "Not verified - Kaggle licence listed as Unknown",
        "status": "Prepared; quality review pending (Week 4)",
    })
manifest = save_manifest(lineage_records)
print("COMMON LINEAGE MANIFEST GENERATED. Entries:", len(manifest["records"]))

Saved lineage: data/week3_manifest.json
COMMON LINEAGE MANIFEST GENERATED. Entries: 1


### Verify the manifest

In [56]:
# VERIFY COMMON LINEAGE MANIFEST
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved_records = json.loads(manifest_path.read_text(encoding="utf-8")).get("records", [])
assert saved_records and len(saved_records) == len(lineage_entries)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source_hash(source) == record["source_sha256"], "Source checksum mismatch"
    assert source_hash(output) == record["prepared_sha256"], "Prepared checksum mismatch"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))

COMMON LINEAGE VALIDATION: PASS
Verified records: 1


## Part C: Build the V1 data pipeline
### Common pipeline utilities

Paths in the run record are stored relative to the project folder so the manifest is the same on any computer.

In [57]:
# REQUIRED: Common V1 Pipeline Utilities
import platform

PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def rel(p):
    return Path(p).resolve().relative_to(BASE_DIR.resolve()).as_posix()

def resolve_source(relative_path):
    source = RAW_DIR / relative_path
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    inputs = [Path(source)]                                   # INGEST
    if any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    inspection = inspect_fn(inputs, configuration)            # INSPECT
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)  # PREPARE / STORE
    outputs = [Path(p) for p in outputs]
    if not outputs:                                           # VALIDATE
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }
    record = {                                                # RECORD LINEAGE
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "environment": {"python": platform.python_version(), "pandas": pd.__version__},
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": rel(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [rel(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified - collection period and version undocumented",
        "access_rights": "Not verified - Kaggle licence listed as Unknown",
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", rel(manifest_path))
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Ready.")

Ready.


### Run the tabular pipeline

In [58]:
# CONDITIONAL REQUIRED: CSV Projects
EXPECTED_COLUMNS = [
    "employee_id", "department", "region", "education", "gender", "recruitment_channel",
    "no_of_trainings", "age", "previous_year_rating", "length_of_service", "awards_won",
    "avg_training_score", "is_promoted",
]

# Step 1: Source and configuration (same as Part B, Configuration A)
SOURCE_B = resolve_source("employee_promotion.csv")
CONFIG_B = {"remove_exact_duplicates": True, "trim_text": True, "impute_missing": False}

# Step 2: Inspect (includes a schema check)
def inspect_table(inputs, cfg):
    data = pd.read_csv(inputs[0])
    if data.empty:
        raise ValueError("Empty input table")
    if data.columns.tolist() != EXPECTED_COLUMNS:
        raise ValueError("Columns do not match the expected schema")
    return {
        "rows": len(data),
        "columns": data.columns.tolist(),
        "missing": {str(k): int(v) for k, v in data.isna().sum().items()},
        "exact_duplicates": int(data.duplicated().sum()),
        "promoted": int(data["is_promoted"].sum()),
    }

# Step 3: Prepare
def prepare_table(inputs, out, cfg):
    src = inputs[0]
    result = pd.read_csv(src)
    if cfg["remove_exact_duplicates"]:
        result = result.drop_duplicates().copy()
    if cfg["trim_text"]:
        for col in result.select_dtypes(include=["object", "string"]).columns:
            result[col] = result[col].astype("string").str.strip()
    if result.empty:
        raise ValueError("No prepared table rows")
    if len(result) != 54808:
        print("WARNING: record count changed - check before continuing")
    target = out / "prepared_table.csv"
    result.to_csv(target, index=False, lineterminator="\n")
    method = (f"remove_exact_duplicates={cfg['remove_exact_duplicates']}; "
              f"trim_text={cfg['trim_text']}; impute_missing={cfg['impute_missing']}")
    return [target], [{"source": rel(src), "output": rel(target)}], method

run_B = register_and_run("tabular", SOURCE_B, inspect_table, prepare_table, CONFIG_B)
print("Prepared output identical to Part B output:",
      run_B["validation"]["output_sha256"][0] == source_hash(prepared_path))
print("Completed successfully.")

SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: data/pipeline_v1/tabular/pipeline_manifest.json
Prepared output identical to Part B output: True
Completed successfully.


### Repeatability and provenance verification

In [59]:
# REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: data/pipeline_v1/tabular/pipeline_manifest.json
PASS: tabular - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
